## 4. Evaluation & Refinement

In this section, we extend the baseline Linear Regression model by exploring regularization techniques and hyperparameter tuning.

The objective is to evaluate whether Ridge and Lasso regression can improve generalization performance and reduce potential multicollinearity effects observed in the baseline model.

We compare model using consistent train/test splits and standarized features, followed by hyperparameter optimization with GridSearchCV. The best-performing model is selected based on evaluation metrics and diagnostic analysis

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/raw/house_data.csv")
df["date"] = pd.to_datetime(df["date"])
df["log_price"] = np.log(df["price"])

### 4.1 Define x/y

In [3]:
# Target
y = df["log_price"]
# Full numeric feature set
x = (df.select_dtypes(include=["int64", "float64"]).drop(columns=["price", "log_price", "id"]))

x.shape, y.shape

((21613, 18), (21613,))

In this step, the target variable (`y`) and feature matrix (`x`) were defined.
The target variable is the **log-transformed house price (`log_price`)**.

The feature set includes all numeric variables except:
 * `price` (original scale target),
 * `log_price` (target variable),
 * `id` (non-informative identifier).
 Using the full numeric feature set allows us to evaluate how regularization techniques (Ridge and Lasso) handle potencial milticollinearity automatically.

### 4.2 Train/Test Split

In [4]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

x_train.shape, x_test.shape

((17290, 18), (4323, 18))

The dataset was split into training and testing sets using an **80/20 ratio**:
- **80%** of the data is used for model training.
- **20%** is reserved for evaluation on unseen data.

A fixed `random_state=42` ensures reproducibility and allows fair comparison between models.
This approach enables objective evaluation of model performance and helps detect potential overfitting.

### 4.3 Build Pipelines (Scaling!)

Linear models are sensitive to feature scale. To ensure fair comparison between Linear Regression, Ridge, and Lasso, all features are standardized using `StandardScaler`.

Pipelines are used to:
- Prevent data leakage
- Ensure scaling is applied  consistently
- Keep the workflow clean and reproducible

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso

#Baseline Linear Regression
pipe_lr = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])

#Ridge Regression
pipe_ridge = Pipeline([("scaler", StandardScaler()), ("model", Ridge())])

#Lasso Regression
pipe_lasso = Pipeline([("scaler", StandardScaler()), ("model", Lasso(max_iter=10000))])

### 4.4 Baseline vs Ridge vs Lasso

In [8]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Fit models
pipe_lr.fit(x_train, y_train)
pipe_ridge.fit(x_train, y_train)
pipe_lasso.fit(x_train, y_train)

# Predictions
y_pred_lr = pipe_lr.predict(x_test)
y_pred_ridge = pipe_ridge.predict(x_test)
y_pred_lasso = pipe_lasso.predict(x_test)

# Collect results
results = pd.DataFrame({"Model": ["Linear Regression", "Ridge", "Lasso"], "R2": [r2_score(y_test, y_pred_lr), r2_score(y_test, y_pred_ridge), r2_score(y_test, y_pred_lasso)], "MAE": [mean_absolute_error(y_test, y_pred_lr), mean_absolute_error(y_test, y_pred_ridge), mean_absolute_error(y_test, y_pred_lasso)], "RMSE": [np.sqrt(mean_squared_error(y_test, y_pred_lr)), np.sqrt(mean_squared_error(y_test, y_pred_ridge)), np.sqrt(mean_squared_error(y_test, y_pred_lasso))]})

results

,Model,R2,MAE,RMSE
0,Linear Regression,0.770980,0.196405,0.255496
1,Ridge,0.770980,0.196405,0.255496
2,Lasso,-0.000617,0.418673,0.534048


### 4.5 Hyperparameter Tuning (GridSearchCV)

### 4.6 Best Model Evaluation

### 4.7 Coefficients/Feature Importance

### 4.8 Final Summary